In [1]:
import os, json, hashlib, datetime as dt
import pandas as pd, requests
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

load_dotenv()
PG_URI = f"postgresql+psycopg2://{os.getenv('PG_USER')}:{os.getenv('PG_PASSWORD')}" \
         f"@{os.getenv('PG_HOST')}:{os.getenv('PG_PORT')}/{os.getenv('PG_DB')}"
engine = create_engine(PG_URI)

In [2]:
# 載入 .env
load_dotenv()

# Database
PG_HOST = os.getenv("PG_HOST")
PG_PORT = os.getenv("PG_PORT")
PG_DB = os.getenv("PG_DB")
PG_USER = os.getenv("PG_USER")
PG_PASSWORD = os.getenv("PG_PASSWORD")

engine = create_engine(
    f"postgresql+psycopg2://{PG_USER}:{PG_PASSWORD}@{PG_HOST}:{PG_PORT}/{PG_DB}"
)

# API keys
FINMIND_TOKEN = os.getenv("FINMIND_TOKEN")
NEWSAPI_KEY = os.getenv("NEWSAPI_KEY")

In [ ]:
# ---------- 台股日價 ----------
def job_stock_dailyprice(symbols:list[str], date:str|None=None):
    api_name = "tw_dailyprice"
    date = (date or dt.date.today().isoformat())

    for sid in symbols:
        params = {"symbol":sid, "date":date}
        params["cache_key"] = f"{sid}-{date}"
        ck = params["cache_key"]

        payload = read_cache(api_name, ck, 1)
        if not payload:
            try:
                # TODO: 換成你的實際 API
                # resp = requests.get(URL, params=params, timeout=20); resp.raise_for_status()
                # payload = resp.json()
                payload = {"data":[{"trade_date":date,"stock_id":sid,"open":100,"high":105,"low":98,"close":102,"volume":123456}]}
                write_cache(api_name, params, 200, payload)
            except Exception as e:
                payload = read_cache(api_name, ck, 7)
                if not payload:
                    log_api(api_name, "FAIL", str(e), params); continue

        # TODO: 依你的 payload 解析
        df = pd.DataFrame(payload["data"])
        df["trade_date"] = pd.to_datetime(df["trade_date"]).dt.date
        upsert("stock_dailyprice", df, ["trade_date","stock_id"])

In [ ]:
def cache_key(api_name:str, params:dict)->str:
    base = api_name + '|' + json.dumps(params, sort_keys=True, ensure_ascii=False)
    return hashlib.md5(base.encode()).hexdigest()

def read_cache(api_name, ck, max_age_days=1):
    sql = """
    SELECT payload FROM fin_raw
    WHERE api_name=:a AND params->>'cache_key'=:ck
      AND fetched_at >= now() - (:age||' days')::interval
    ORDER BY fetched_at DESC LIMIT 1
    """
    with engine.begin() as c:
        r = c.execute(text(sql), {"a":api_name,"ck":ck,"age":max_age_days}).fetchone()
        return r[0] if r else None

def write_cache(api_name, params, status, payload):
    with engine.begin() as c:
        c.execute(text("""
            INSERT INTO fin_raw(api_name, params, status, payload)
            VALUES (:a, :p::jsonb, :s, :pl::jsonb)
            ON CONFLICT (api_name, (params->>'cache_key'), date_trunc('day', fetched_at))
            DO UPDATE SET status=EXCLUDED.status, payload=EXCLUDED.payload, fetched_at=now()
        """), {"a":api_name, "p":json.dumps(params,ensure_ascii=False),
               "s":status, "pl":json.dumps(payload,ensure_ascii=False)})

def upsert(table:str, df:pd.DataFrame, pk:list[str]):
    if df.empty: return
    cols = list(df.columns)
    values = ", ".join([f":{c}" for c in cols])
    set_ = ", ".join([f"{c}=EXCLUDED.{c}" for c in cols if c not in pk])
    sql = f"INSERT INTO {table} ({', '.join(cols)}) VALUES ({values}) " \
          f"ON CONFLICT ({', '.join(pk)}) DO UPDATE SET {set_}"
    with engine.begin() as c:
        c.execute(text(sql), df.to_dict(orient="records"))

def log_api(name, status, msg, params):
    with engine.begin() as c:
        c.execute(text("INSERT INTO api_log(api_name,status,message,params) VALUES(:n,:s,:m,:p::jsonb)"),
                  {"n":name,"s":status,"m":msg,"p":json.dumps(params)})

# ---------- 例1：台股日價 ----------
def job_stock_dailyprice(symbols:list[str], date:str|None=None):
    api_name = "tw_dailyprice"
    date = (date or dt.date.today().isoformat())

    for sid in symbols:
        params = {"symbol":sid, "date":date}
        params["cache_key"] = f"{sid}-{date}"
        ck = params["cache_key"]

        payload = read_cache(api_name, ck, 1)
        if not payload:
            try:
                # TODO: 換成你的實際 API
                # resp = requests.get(URL, params=params, timeout=20); resp.raise_for_status()
                # payload = resp.json()
                payload = {"data":[{"trade_date":date,"stock_id":sid,"open":100,"high":105,"low":98,"close":102,"volume":123456}]}
                write_cache(api_name, params, 200, payload)
            except Exception as e:
                payload = read_cache(api_name, ck, 7)
                if not payload:
                    log_api(api_name, "FAIL", str(e), params); continue

        # TODO: 依你的 payload 解析
        df = pd.DataFrame(payload["data"])
        df["trade_date"] = pd.to_datetime(df["trade_date"]).dt.date
        upsert("stock_dailyprice", df, ["trade_date","stock_id"])

# ---------- 例2：匯率 ----------
def job_exchange_rate(quotes:list[str], base="TWD", date:str|None=None):
    api_name = "fx_rate"
    date = (date or dt.date.today().isoformat())

    for q in quotes:
        params = {"base":base,"quote":q,"date":date}
        params["cache_key"] = f"{base}-{q}-{date}"
        ck = params["cache_key"]

        payload = read_cache(api_name, ck, 1)
        if not payload:
            try:
                # TODO: 換成你的實際 API
                payload = {"rate": 0.0321}
                write_cache(api_name, params, 200, payload)
            except Exception as e:
                payload = read_cache(api_name, ck, 7)
                if not payload:
                    log_api(api_name, "FAIL", str(e), params); continue

        df = pd.DataFrame([{
            "rate_date": date, "base_ccy": base, "quote_ccy": q, "rate": payload["rate"]
        }])
        df["rate_date"] = pd.to_datetime(df["rate_date"]).dt.date
        upsert("exchange_rate", df, ["rate_date","quote_ccy"])

if __name__ == "__main__":
    # 從主檔抓股票清單（先測少量）
    with engine.begin() as c:
        syms = [r[0] for r in c.execute(text("SELECT stock_id FROM stock_info WHERE list_type IN ('twse','tpex','etf') LIMIT 50"))]
    job_stock_dailyprice(syms)
    job_exchange_rate(["USD","EUR","JPY"])
